# ML-10 — Content Action Playbook

This notebook turns the measured ranking into a queue for human review.


## 1. Ranked actions and reason codes

This review queue uses the simple visibility and CTR rule on observed June search data. On the May-to-June held-out-client check, that rule ranked declines better than the logistic model at the top 20. A higher rule score means higher review priority, not a measured chance that a refresh will work. Each page gets a reason code so an editor can check the suggestion.


## Load the May and June data

I keep the large daily records in DuckDB and bring only page-level results into pandas. The first run saves the results locally. The daily batches keep memory use down.


In [10]:
import getpass
import os
import tempfile
from datetime import date, timedelta
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

FEATURE_COLUMNS = (
    'log_prior30_impressions', 'log_prior30_clicks', 'prior30_ctr',
    'prior30_avg_position_filled', 'position_missing',
)

LABEL_END = date(2026, 6, 30)
HISTORICAL_CUTOFF = LABEL_END - timedelta(days=30)
FEATURE_START = HISTORICAL_CUTOFF - timedelta(days=29)
CURRENT_START = HISTORICAL_CUTOFF + timedelta(days=1)

def repo_root() -> Path:
    for path in (Path.cwd(), *Path.cwd().parents):
        if (path / 'work' / 'notebooks').is_dir():
            return path
    raise FileNotFoundError('Open the repository or a folder inside it before running the notebook')

def _quote_path(path: Path) -> str:
    return str(path).replace('\\', '/').replace("'", "''")

def _feature_select(table: str, labeled: bool) -> str:
    extra = (
        ", future30_impressions, CASE WHEN future30_impressions < 0.8 * prior30_impressions "
        "THEN 1 ELSE 0 END AS future_decline_label"
    ) if labeled else ''
    return f"""
        SELECT client_hash_id, content_hash_id, prior30_impressions, prior30_clicks,
               prior30_avg_position,
               LN(1 + prior30_impressions) AS log_prior30_impressions,
               LN(1 + prior30_clicks) AS log_prior30_clicks,
               CAST(prior30_clicks AS DOUBLE) / prior30_impressions AS prior30_ctr,
               COALESCE(prior30_avg_position, 0.0) AS prior30_avg_position_filled,
               CASE WHEN prior30_avg_position IS NULL THEN 1 ELSE 0 END AS position_missing
               {extra}
        FROM {table}
    """

repo = repo_root()


In [11]:
def _summarize_month(con: duckdb.DuckDBPyConnection, source: str,
                     start: date, end: date, folder: Path) -> Path:
    """Stream a month to disk, then deduplicate one day at a time."""
    folder.mkdir()
    stage = folder / 'daily_source.parquet'
    days = folder / 'days'
    days.mkdir()
    summary = folder / 'summary.parquet'
    con.execute(f"""
        COPY (SELECT report_date, client_hash_id, content_hash_id,
                     gsc_impressions, gsc_clicks, gsc_avg_position
              FROM {source}
              WHERE report_date BETWEEN DATE '{start}' AND DATE '{end}')
        TO '{_quote_path(stage)}' (FORMAT PARQUET)
    """)
    day = start
    while day <= end:
        day_path = days / f'{day}.parquet'
        con.execute(f"""
            COPY (
                SELECT client_hash_id, content_hash_id,
                       MIN(gsc_impressions) AS impressions,
                       MIN(gsc_clicks) AS clicks,
                       CASE WHEN MIN(gsc_avg_position) > 0 AND MIN(gsc_impressions) > 0
                            THEN MIN(gsc_avg_position) * MIN(gsc_impressions)
                            ELSE 0 END AS position_numerator,
                       CASE WHEN MIN(gsc_avg_position) > 0 AND MIN(gsc_impressions) > 0
                            THEN MIN(gsc_impressions) ELSE 0 END AS position_denominator,
                       COUNT(*) - 1 AS repeated_rows,
                       CASE WHEN MIN(gsc_impressions) IS DISTINCT FROM MAX(gsc_impressions)
                                      OR MIN(gsc_clicks) IS DISTINCT FROM MAX(gsc_clicks)
                                      OR MIN(gsc_avg_position) IS DISTINCT FROM MAX(gsc_avg_position)
                                      OR (COUNT(gsc_impressions) > 0 AND COUNT(gsc_impressions) < COUNT(*))
                                      OR (COUNT(gsc_clicks) > 0 AND COUNT(gsc_clicks) < COUNT(*))
                                      OR (COUNT(gsc_avg_position) > 0 AND COUNT(gsc_avg_position) < COUNT(*))
                            THEN 1 ELSE 0 END AS conflicting_keys
                FROM read_parquet('{_quote_path(stage)}')
                WHERE report_date = DATE '{day}'
                GROUP BY client_hash_id, content_hash_id
            ) TO '{_quote_path(day_path)}' (FORMAT PARQUET)
        """)
        day += timedelta(days=1)
    stage.unlink()
    con.execute(f"""
        COPY (
            SELECT client_hash_id, content_hash_id, COUNT(*) AS observed_days,
                   COUNT(*) FILTER (WHERE impressions IS NULL OR clicks IS NULL) AS missing_days,
                   SUM(impressions) AS impressions, SUM(clicks) AS clicks,
                   SUM(position_numerator) / NULLIF(SUM(position_denominator), 0) AS avg_position,
                   SUM(repeated_rows) AS repeated_rows,
                   SUM(conflicting_keys) AS conflicting_keys
            FROM read_parquet('{_quote_path(days / '*.parquet')}')
            GROUP BY client_hash_id, content_hash_id
        ) TO '{_quote_path(summary)}' (FORMAT PARQUET)
    """)
    for day_path in days.glob('*.parquet'):
        day_path.unlink()
    days.rmdir()
    return summary


In [12]:
def build_frames(source_override: str | None = None,
                 connection: duckdb.DuckDBPyConnection | None = None
                 ) -> tuple[pd.DataFrame, pd.DataFrame]:
    output = repo_root() / 'work' / 'outputs'
    output.mkdir(parents=True, exist_ok=True)
    con = connection or duckdb.connect()
    with tempfile.TemporaryDirectory(prefix='lane2_build_', dir=output) as scratch:
        temp = Path(scratch)
        spill = output / 'lane2_duckdb_spill'
        spill.mkdir(exist_ok=True)
        con.execute(f"SET temp_directory = '{_quote_path(spill)}'")
        con.execute("SET memory_limit = '1GB'")
        con.execute('SET threads = 2')
        if source_override is None:
            token = os.environ.get('HF_TOKEN') or getpass.getpass('Hugging Face READ token: ')
            con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '"
                        + token.replace("'", "''") + "')")
            del token
            root = 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance'
            may_source = f"read_parquet('{root}/month=2026-05/*.parquet')"
            june_source = f"read_parquet('{root}/month=2026-06/*.parquet')"
        else:
            may_source = june_source = source_override

        may = _summarize_month(con, may_source, FEATURE_START, HISTORICAL_CUTOFF,
                               temp / 'may')
        june = _summarize_month(con, june_source, CURRENT_START, LABEL_END,
                                temp / 'june')
        may_rel = f"read_parquet('{_quote_path(may)}')"
        june_rel = f"read_parquet('{_quote_path(june)}')"
        counts = con.sql(f"""
            SELECT SUM(repeated_rows), SUM(conflicting_keys),
                   COUNT(*) FILTER (WHERE client_hash_id IS NULL OR content_hash_id IS NULL)
            FROM (SELECT * FROM {may_rel} UNION ALL SELECT * FROM {june_rel})
        """).fetchone()
        repeated, conflicting, missing_keys = (int(value or 0) for value in counts)
        assert missing_keys == 0, f'{missing_keys:,} rows have a missing daily key'
        assert conflicting == 0, f'{conflicting:,} daily keys have different GSC values'

        con.execute(f"""
            CREATE OR REPLACE TEMP TABLE historical_metrics AS
            SELECT m.client_hash_id, m.content_hash_id,
                   m.impressions AS prior30_impressions,
                   m.clicks AS prior30_clicks,
                   m.avg_position AS prior30_avg_position,
                   j.impressions AS future30_impressions
            FROM {may_rel} m
            JOIN {june_rel} j USING (client_hash_id, content_hash_id)
            WHERE m.observed_days = 30 AND m.missing_days = 0
              AND m.impressions >= 100
              AND j.observed_days = 30 AND j.missing_days = 0
        """)
        con.execute(f"""
            CREATE OR REPLACE TEMP TABLE current_metrics AS
            SELECT client_hash_id, content_hash_id,
                   impressions AS prior30_impressions,
                   clicks AS prior30_clicks,
                   avg_position AS prior30_avg_position
            FROM {june_rel}
            WHERE observed_days = 30 AND missing_days = 0 AND impressions >= 100
        """)
        historical = con.sql(_feature_select('historical_metrics', labeled=True)).df()
        current = con.sql(_feature_select('current_metrics', labeled=False)).df()

    assert len(historical) > 0 and len(current) > 0, 'No pages met the coverage and impression rules'
    assert historical[list(FEATURE_COLUMNS)].notna().all().all()
    assert current[list(FEATURE_COLUMNS)].notna().all().all()
    print(f'May-June repeated GSC rows removed: {repeated:,}')
    print(f'Historical labeled pages: {len(historical):,}; current June pages: {len(current):,}')
    return historical, current

def load_frames() -> tuple[pd.DataFrame, pd.DataFrame]:
    output = repo_root() / 'work' / 'outputs'
    history_path = output / 'lane2_history_2026-05-31.parquet'
    current_path = output / 'lane2_current_2026-06-30.parquet'
    if history_path.exists() and current_path.exists():
        con = duckdb.connect()
        history = con.sql(f"SELECT * FROM read_parquet('{_quote_path(history_path)}')").df()
        current = con.sql(f"SELECT * FROM read_parquet('{_quote_path(current_path)}')").df()
        print(f'Loaded cached frames: {len(history):,} historical, {len(current):,} current pages')
        return history, current
    history, current = build_frames()
    output.mkdir(parents=True, exist_ok=True)
    con = duckdb.connect()
    con.register('cache_history', history)
    con.register('cache_current', current)
    con.execute(f"COPY cache_history TO '{_quote_path(history_path)}' (FORMAT PARQUET)")
    con.execute(f"COPY cache_current TO '{_quote_path(current_path)}' (FORMAT PARQUET)")
    print('Saved local Parquet caches under work/outputs; dataset files are gitignored')
    return history, current


In [13]:
def grouped_split(frame: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    groups = frame['client_hash_id']
    if groups.nunique() < 2:
        raise ValueError('A grouped split needs at least two clients')
    train, test = next(GroupShuffleSplit(n_splits=1, test_size=0.25,
                                         random_state=42).split(frame, groups=groups))
    assert set(groups.iloc[train]).isdisjoint(set(groups.iloc[test]))
    return train, test

def baseline_settings(frame: pd.DataFrame, train: np.ndarray) -> tuple[float, float]:
    part = frame.iloc[train]
    return float(part['prior30_impressions'].median()), float(part['prior30_ctr'].quantile(0.25))

def baseline_rank(frame: pd.DataFrame, settings: tuple[float, float]) -> pd.DataFrame:
    visibility_min, low_ctr_max = settings
    visible = frame['prior30_impressions'] >= visibility_min
    low_ctr = frame['prior30_ctr'] <= low_ctr_max
    striking = frame['prior30_avg_position'].between(11, 20, inclusive='both')
    result = frame[['client_hash_id', 'content_hash_id', 'prior30_impressions',
                    'prior30_clicks', 'prior30_ctr', 'prior30_avg_position', 'position_missing']].copy()
    result['baseline_score'] = (visible.astype(int) * (2 + low_ctr.astype(int) + striking.astype(int))
                                + 0.01 * np.log1p(frame['prior30_impressions']))
    result['reason_code'] = np.select(
        [visible & low_ctr & striking, visible & low_ctr, visible & striking, visible],
        ['visible_low_ctr_striking', 'visible_low_ctr', 'visible_striking', 'visible_page'],
        default='low_visibility_review_later',
    )
    return result

def precision_at_k(scores, labels, k: int) -> tuple[float, int]:
    values = np.asarray(scores, dtype=float)
    truth = np.asarray(labels, dtype=int)
    if len(values) != len(truth) or len(values) == 0:
        raise ValueError('Scores and labels need the same nonzero length')
    used = min(k, len(values))
    order = np.argsort(-values, kind='stable')[:used]
    return float(truth[order].mean()), used


In [ ]:
from pathlib import Path
import numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

history, current = load_frames()
features = list(FEATURE_COLUMNS)
y = history.future_decline_label.astype(int)
train_idx, test_idx = grouped_split(history)
settings = baseline_settings(history, train_idx)
check_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42))
check_model.fit(history[features].iloc[train_idx], y.iloc[train_idx])
model_scores = check_model.predict_proba(history[features].iloc[test_idx])[:, 1]
model_precision, heldout_k = precision_at_k(model_scores, y.iloc[test_idx], 20)
rule_scores = baseline_rank(history, settings).baseline_score.iloc[test_idx]
heldout_precision, _ = precision_at_k(rule_scores, y.iloc[test_idx], 20)
heldout_base = y.iloc[test_idx].mean()
print(f'Historical held-out-client precision at {heldout_k}: '
      f'rule {heldout_precision:.1%}; model {model_precision:.1%}; '
      f'decline base rate {heldout_base:.1%}')
queue = baseline_rank(current, settings)

def action_for(reason):
    if 'low_ctr' in reason:
        return 'Review title, snippet, and search intent'
    if 'striking' in reason:
        return 'Review relevance and internal links'
    if reason == 'visible_page':
        return 'Check content accuracy and freshness'
    return 'Gather more evidence before changing the page'

queue['suggested_action'] = queue.reason_code.map(action_for)
queue['human_review_required'] = True
queue = queue.sort_values('baseline_score', ascending=False, kind='stable').reset_index(drop=True)
print(f'Pages in the June review queue: {len(queue):,}')
for rank, row in enumerate(queue.head(20).itertuples(index=False), start=1):
    print(f'{rank:2d}. {row.suggested_action}. Reason: {row.reason_code}. '
          f'Review score: {row.baseline_score:.2f}.')

## 2. Intended use and limits

An editor uses this queue to decide which pages to inspect first. The rule uses June search signals and was checked on held-out clients at the earlier May cutoff. Pages without 30 observed June days or with fewer than 100 June impressions are outside this queue. July outcomes are not available, and the rule has not been tested at a later cutoff. This is decision support, not an instruction to refresh every listed page.


In [ ]:
print('Scoring date: 30 June 2026')
print(f'Eligible June pages: {len(current):,}')
print(f'June pages with missing average position: {int(current.position_missing.sum()):,}')
print('July outcome check: unavailable in this release')
assert 'future_decline_label' not in current.columns
assert queue.human_review_required.all()

Scoring date: 30 June 2026
Eligible June pages: 96,678
June pages with missing average position: 0
July outcome check: unavailable in this release


## 3. Human review and no-go list

Before changing a page, an editor checks the page itself, its search intent inside an approved tool, recent edits, seasonality, and whether the suggested action fits. The editor records a decision and reason. The queue must not publish edits, rewrite pages automatically, contact clients, or treat the score as proof of traffic improvement. Missing position or weak supporting evidence needs extra care.

In [ ]:
review_steps = [
    'Check page accuracy and current search intent',
    'Check recent edits and seasonal demand',
    'Check whether the reason code matches what the page needs',
    'Record an editor decision before any change',
]
for step in review_steps:
    print('Human check:', step)
print(f'Top-20 pages with missing position: {int(queue.head(20).position_missing.sum())}')
print('Automatic publishing and client contact are not permitted by this queue')

Human check: Check page accuracy and current search intent
Human check: Check recent edits and seasonal demand
Human check: Check whether the reason code matches what the page needs
Human check: Record an editor decision before any change
Top-20 pages with missing position: 0
Automatic publishing and client contact are not permitted by this queue


## 4. Monitoring and retraining triggers

The first future performance check needs July data. I would compare the rule's precision at 20 with the July decline base rate on pages eligible at the June cutoff. For an early input check, I compare May and June CTR only on pages eligible in both windows, so a change in which pages are included does not drive the whole result. This still excludes pages that fell below the June eligibility threshold. A large shift calls for review before using the queue; retraining needs a checked later window.


In [ ]:
matched = history[['client_hash_id', 'content_hash_id', 'prior30_ctr']].merge(
    current[['client_hash_id', 'content_hash_id', 'prior30_ctr']],
    on=['client_hash_id', 'content_hash_id'],
    suffixes=('_may', '_june')
)
assert len(matched) > 0
history_ctr_median = matched.prior30_ctr_may.median()
current_ctr_median = matched.prior30_ctr_june.median()
ctr_shift = abs(current_ctr_median - history_ctr_median) / max(abs(history_ctr_median), 1e-9)
missing_position_rate = current.position_missing.mean()
print(f'Pages eligible in both windows: {len(matched):,}')
print(f'Median CTR shift on matched pages: {ctr_shift:.1%}')
print(f'Current pages with missing position: {missing_position_rate:.1%}')
print('Review trigger: matched-page median CTR shift above 25% or missing-position rate above 10%')
print('Performance trigger after July labels exist: rule precision at 20 below the July base rate')
if ctr_shift > 0.25 or missing_position_rate > 0.10:
    print('Current review trigger: investigate before using the queue')
else:
    print('Current review trigger: not reached by these two checks')


Pages eligible in both windows: 83,012
Median CTR shift on matched pages: 20.3%
Current pages with missing position: 0.0%
Review trigger: matched-page median CTR shift above 25% or missing-position rate above 10%
Performance trigger after July labels exist: rule precision at 20 below the July base rate
Current review trigger: not reached by these two checks


## 5. Exports for the paper

The local queue export has pseudonymous IDs, search-only evidence, the rule score, a reason code, and a suggested action. It has no client names, page URLs, raw queries, or future label. The text export gives aggregate counts and the held-out comparison for the paper. Both describe a review process, not measured refresh impact.


In [ ]:
output_dir = repo / 'work' / 'outputs'
output_dir.mkdir(parents=True, exist_ok=True)
queue_path = output_dir / 'action_playbook_queue.csv'
summary_path = output_dir / 'action_playbook_summary.txt'
forbidden = ('future', 'label', 'url', 'query', 'ga4', 'client_name', 'domain')
assert not any(word in column for column in queue.columns for word in forbidden)
queue.to_csv(queue_path, index=False)
summary = (
    'Lane 2 review queue as of 30 June 2026\n'
    f'Eligible pages: {len(queue):,}\n'
    f'Historical held-out-client rule precision at {heldout_k}: {heldout_precision:.1%}\n'
    f'Historical held-out-client model precision at {heldout_k}: {model_precision:.1%}\n'
    f'Historical decline base rate: {heldout_base:.1%}\n'
    'July outcomes have not been observed in this release.\n'
    'Scores rank pages for human review and do not measure refresh impact.\n'
)
summary_path.write_text(summary, encoding='utf-8')
print(f'Saved queue: {queue_path}')
print(f'Saved aggregate summary: {summary_path}')

Saved queue: D:\flyrank-ml-internship-starter\work\outputs\action_playbook_queue.csv
Saved aggregate summary: D:\flyrank-ml-internship-starter\work\outputs\action_playbook_summary.txt


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under work/notebooks/ — then submit your repo URL on the card. Done.